In [ ]:
import pickle
import together
from itertools import product
import time
from config.settings import TOGETHER_API_KEY

together_client = together.Together(api_key=TOGETHER_API_KEY)

epochs_list = [1, 3, 6]
r_list = [8, 16, 32]
poll_interval = 60  # seconds between status checks

## Fine-tuning via Together AI

In [ ]:
# Upload fine-tuning data
resp = together_client.files.upload(file="data/duch_et_al_2023_training_1538/duch_et_al_2023_training_1538_train.jsonl", purpose="fine-tune")
print(f"File ID: {resp.id}")

In [ ]:
ft_jobs = {}
for n_epochs, r in product(epochs_list, r_list):
    alpha = 2 * r
    suffix = f"duch2023_ep{n_epochs}_r{r}"

    ft_resp = together_client.fine_tuning.create(
        training_file=resp.id,
        model="meta-llama/Meta-Llama-3.1-70B-Instruct-Reference",
        n_epochs=n_epochs,
        n_checkpoints=1,
        n_evals=0,
        batch_size=8,
        learning_rate=2e-5,
        lr_scheduler_type="linear",
        warmup_ratio=0.1,
        max_grad_norm=1,
        weight_decay=0,
        lora_r=r,
        lora_alpha=alpha,
        lora_dropout=0.05,
        lora_trainable_modules="all-linear",
        train_on_inputs="auto",
        suffix=suffix
    )

    key = f"duch2023_ep{n_epochs}_r{r}"
    ft_jobs[key] = ft_resp.id
    print(f"Created job: {key} -> {ft_resp.id}")

# Save jobs for reference
with open("llama3_1_70b_duch2023_ft_jobs.pkl", "wb") as f:
    pickle.dump(ft_jobs, f)
print(f"\nTotal jobs created: {len(ft_jobs)}")

In [ ]:
ft_models = {}
while True:
    all_done = True
    for r, job_id in ft_jobs.items():
        if r in ft_models:
            continue  # already completed
        ft_status = together_client.fine_tuning.retrieve(id=job_id)
        status_str = str(ft_status.status)
        print(f"  r={r} (job {job_id}): {status_str}")

        if "COMPLETED" in status_str.upper():
            ft_models[r] = ft_status.model_output_name
            print(f"    -> Model: {ft_status.model_output_name}")
        elif "FAILED" in status_str.upper() or "CANCELLED" in status_str.upper():
            print(f"    -> Job failed/cancelled!")
            ft_models[r] = None
        else:
            all_done = False

    if all_done:
        print("\nAll fine-tuning jobs finished!")
        break

    print(f"\nWaiting {poll_interval}s before next check...\n")
    time.sleep(poll_interval)

print("\nFine-tuned models:")
for r, model_name in ft_models.items():
    print(f"  r={r}: {model_name}")